## Deep Neural Network for Asteroid Diameter Prediction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)

In [ ]:
# Load and prepare data
filepath = '/content/drive/MyDrive/Data/dataset.csv'
df = pd.read_csv(filepath)

# Define columns to exclude
exclude_cols = [
    'id', 'spkid', 'full_name', 'pdes', 'name', 'prefix', 'orbit_id',
    'diameter', 'diameter_sigma', 'epoch', 'epoch_cal', 'tp_cal',
    'per', 'per_y',
    'sigma_e', 'sigma_a', 'sigma_q', 'sigma_i', 'sigma_om',
    'sigma_w', 'sigma_ma', 'sigma_ad', 'sigma_n', 'sigma_tp', 'sigma_per'
]

df_clean = df.drop(columns=exclude_cols, errors='ignore')

# Encode categorical variables
df_clean['neo'] = df_clean['neo'].map({'Y': 1, 'N': 0})
df_clean['pha'] = df_clean['pha'].map({'Y': 1, 'N': 0})
df_clean = pd.get_dummies(df_clean, columns=['equinox'], drop_first=True)

# Remove missing values
df_final = df_clean.dropna()

# Separate features and target
X = df_final.drop(columns=['diameter'])
y = df_final['diameter']

# Remove any non-numeric columns
X = X.select_dtypes(include=[np.number])

print(f"Dataset prepared:")
print(f"  X shape: {X.shape}")
print(f"  y shape: {y.shape}")
print(f"  Features: {list(X.columns)}")

In [ ]:
# Split data into train, validation, and test sets
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

print(f"Data split:")
print(f"  Train: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"  Validation: {X_val.shape[0]:,} samples ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"  Test: {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"\nFeature scaling completed:")
print(f"  X_train_scaled shape: {X_train_scaled.shape}")
print(f"  X_val_scaled shape: {X_val_scaled.shape}")
print(f"  X_test_scaled shape: {X_test_scaled.shape}")

In [ ]:
# Define DNN architecture with hyperparameters
print("="*70)
print("DNN HYPERPARAMETERS CONFIGURATION")
print("="*70)

# Hyperparameters to tune
lr = 0.001  # Learning rate
batch_size = 32  # Batch size
num_neurons = 128  # Number of neurons in hidden layers
dropout_rate = 0.3  # Dropout rate
epochs = 150

print(f"\n1. LEARNING RATE: {lr}")
print(f"   - Controls step size in gradient descent")
print(f"   - Smaller = slower but more stable learning")
print(f"   - Larger = faster but may overshoot")

print(f"\n2. BATCH SIZE: {batch_size}")
print(f"   - Number of samples per gradient update")
print(f"   - Smaller = noisier gradients but faster training")
print(f"   - Larger = smoother gradients but requires more memory")

print(f"\n3. NUMBER OF NEURONS: {num_neurons}")
print(f"   - Model capacity and expressiveness")
print(f"   - More neurons = more complex patterns but risk of overfitting")
print(f"   - Architecture: {X_train_scaled.shape[1]} → {num_neurons} → {num_neurons} → {num_neurons} → 1")

print(f"\n4. DROPOUT RATE: {dropout_rate}")
print(f"   - Regularization to prevent overfitting")
print(f"   - Randomly deactivates {dropout_rate*100}% of neurons during training")
print(f"   - Helps model generalize better")

In [ ]:
# Build DNN model
model = keras.Sequential([
    # Input layer
    layers.Input(shape=(X_train_scaled.shape[1],)),
    
    # Hidden Layer 1
    layers.Dense(num_neurons, activation='relu', name='dense_1'),
    layers.Dropout(dropout_rate, name='dropout_1'),
    
    # Hidden Layer 2
    layers.Dense(num_neurons, activation='relu', name='dense_2'),
    layers.Dropout(dropout_rate, name='dropout_2'),
    
    # Hidden Layer 3
    layers.Dense(num_neurons, activation='relu', name='dense_3'),
    layers.Dropout(dropout_rate, name='dropout_3'),
    
    # Output layer (regression - single neuron, linear activation)
    layers.Dense(1, activation='linear', name='output')
])

# Compile model
optimizer = Adam(learning_rate=lr)
model.compile(
    optimizer=optimizer,
    loss='mse',  # Mean Squared Error for regression
    metrics=['mae', 'mse']  # Mean Absolute Error
)

# Print model summary
print("="*70)
print("DNN MODEL ARCHITECTURE")
print("="*70)
model.summary()

print("\n" + "="*70)
print("MODEL CONFIGURATION")
print("="*70)
print(f"Optimizer: Adam (learning_rate={lr})")
print(f"Loss Function: Mean Squared Error (MSE)")
print(f"Metrics: MAE, MSE")
print(f"Total Parameters: {model.count_params():,}")

In [ ]:
# Train the model
print("\n" + "="*70)
print("TRAINING THE MODEL")
print("="*70)
print(f"\nTraining configuration:")
print(f"  Epochs: {epochs}")
print(f"  Batch Size: {batch_size}")
print(f"  Training samples: {X_train_scaled.shape[0]:,}")
print(f"  Validation samples: {X_val_scaled.shape[0]:,}")
print(f"  Batches per epoch: {X_train_scaled.shape[0] // batch_size}")

# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True,
    verbose=1
)

# Train model
history = model.fit(
    X_train_scaled, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(X_val_scaled, y_val),
    callbacks=[early_stopping],
    verbose=1
)

print("\nTraining completed!")

In [ ]:
# Plot training and validation loss
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('DNN Training Performance: Loss and MAE', fontsize=14, fontweight='bold')

# Plot 1: Loss
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2, color='blue')
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2, color='red')
axes[0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Loss (MSE)', fontsize=12, fontweight='bold')
axes[0].set_title('Training vs Validation Loss', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=11, loc='upper right')
axes[0].grid(alpha=0.3)

# Add annotations
min_val_loss = min(history.history['val_loss'])
min_val_epoch = history.history['val_loss'].index(min_val_loss)
axes[0].annotate(f'Best: {min_val_loss:.2f}', 
                xy=(min_val_epoch, min_val_loss),
                xytext=(min_val_epoch+10, min_val_loss*1.5),
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
                fontsize=10, fontweight='bold')

# Plot 2: MAE
axes[1].plot(history.history['mae'], label='Training MAE', linewidth=2, color='green')
axes[1].plot(history.history['val_mae'], label='Validation MAE', linewidth=2, color='orange')
axes[1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[1].set_ylabel('MAE (km)', fontsize=12, fontweight='bold')
axes[1].set_title('Training vs Validation MAE', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=11, loc='upper right')
axes[1].grid(alpha=0.3)

# Add annotations
min_val_mae = min(history.history['val_mae'])
min_mae_epoch = history.history['val_mae'].index(min_val_mae)
axes[1].annotate(f'Best: {min_val_mae:.2f}', 
                xy=(min_mae_epoch, min_val_mae),
                xytext=(min_mae_epoch+10, min_val_mae*1.5),
                arrowprops=dict(arrowstyle='->', color='orange', lw=1.5),
                fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
print("\n" + "="*70)
print("MODEL EVALUATION")
print("="*70)

# Predictions on all sets
y_train_pred = model.predict(X_train_scaled, verbose=0).flatten()
y_val_pred = model.predict(X_val_scaled, verbose=0).flatten()
y_test_pred = model.predict(X_test_scaled, verbose=0).flatten()

# Calculate metrics
print("\nTraining Set Metrics:")
train_mse = mean_squared_error(y_train, y_train_pred)
train_mae = mean_absolute_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
print(f"  MSE:  {train_mse:.4f}")
print(f"  RMSE: {train_rmse:.4f} km")
print(f"  MAE:  {train_mae:.4f} km")
print(f"  R²:   {train_r2:.4f}")

print("\nValidation Set Metrics:")
val_mse = mean_squared_error(y_val, y_val_pred)
val_mae = mean_absolute_error(y_val, y_val_pred)
val_r2 = r2_score(y_val, y_val_pred)
val_rmse = np.sqrt(val_mse)
print(f"  MSE:  {val_mse:.4f}")
print(f"  RMSE: {val_rmse:.4f} km")
print(f"  MAE:  {val_mae:.4f} km")
print(f"  R²:   {val_r2:.4f}")

print("\nTest Set Metrics:")
test_mse = mean_squared_error(y_test, y_test_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
print(f"  MSE:  {test_mse:.4f}")
print(f"  RMSE: {test_rmse:.4f} km")
print(f"  MAE:  {test_mae:.4f} km")
print(f"  R²:   {test_r2:.4f}")

print("\n" + "="*70)

In [ ]:
# Visualize predictions vs actual
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Model Predictions vs Actual Values', fontsize=14, fontweight='bold')

# Training set
axes[0].scatter(y_train, y_train_pred, alpha=0.5, s=20, color='blue')
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Diameter (km)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Predicted Diameter (km)', fontsize=11, fontweight='bold')
axes[0].set_title(f'Training Set (R² = {train_r2:.4f})', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)

# Validation set
axes[1].scatter(y_val, y_val_pred, alpha=0.5, s=20, color='green')
axes[1].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual Diameter (km)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Predicted Diameter (km)', fontsize=11, fontweight='bold')
axes[1].set_title(f'Validation Set (R² = {val_r2:.4f})', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3)

# Test set
axes[2].scatter(y_test, y_test_pred, alpha=0.5, s=20, color='orange')
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[2].set_xlabel('Actual Diameter (km)', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Predicted Diameter (km)', fontsize=11, fontweight='bold')
axes[2].set_title(f'Test Set (R² = {test_r2:.4f})', fontsize=12, fontweight='bold')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot residuals
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Residuals Analysis (Predicted - Actual)', fontsize=14, fontweight='bold')

train_residuals = y_train_pred - y_train
val_residuals = y_val_pred - y_val
test_residuals = y_test_pred - y_test

# Training residuals
axes[0].scatter(y_train_pred, train_residuals, alpha=0.5, s=20, color='blue')
axes[0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0].set_xlabel('Predicted Diameter (km)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Residuals (km)', fontsize=11, fontweight='bold')
axes[0].set_title(f'Training Set (Mean: {train_residuals.mean():.4f})', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)

# Validation residuals
axes[1].scatter(y_val_pred, val_residuals, alpha=0.5, s=20, color='green')
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Diameter (km)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Residuals (km)', fontsize=11, fontweight='bold')
axes[1].set_title(f'Validation Set (Mean: {val_residuals.mean():.4f})', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3)

# Test residuals
axes[2].scatter(y_test_pred, test_residuals, alpha=0.5, s=20, color='orange')
axes[2].axhline(y=0, color='r', linestyle='--', lw=2)
axes[2].set_xlabel('Predicted Diameter (km)', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Residuals (km)', fontsize=11, fontweight='bold')
axes[2].set_title(f'Test Set (Mean: {test_residuals.mean():.4f})', fontsize=12, fontweight='bold')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Summary report
print("\n" + "="*70)
print("FINAL SUMMARY REPORT")
print("="*70)

print("\n1. HYPERPARAMETERS USED:")
print(f"   Learning Rate: {lr}")
print(f"   Batch Size: {batch_size}")
print(f"   Number of Neurons: {num_neurons}")
print(f"   Dropout Rate: {dropout_rate}")
print(f"   Epochs Trained: {len(history.history['loss'])}")

print("\n2. MODEL ARCHITECTURE:")
print(f"   Input Layer: {X_train_scaled.shape[1]} features")
print(f"   Hidden Layers: 3 × {num_neurons} neurons with ReLU activation")
print(f"   Dropout: {dropout_rate*100}% applied after each hidden layer")
print(f"   Output Layer: 1 neuron with Linear activation (Regression)")
print(f"   Total Parameters: {model.count_params():,}")

print("\n3. BEST PERFORMANCE:")
print(f"   Best Validation MSE: {min(history.history['val_loss']):.4f}")
print(f"   Best Validation MAE: {min(history.history['val_mae']):.4f} km")
print(f"   Best Epoch: {history.history['val_loss'].index(min(history.history['val_loss']))+1}")

print("\n4. TEST SET RESULTS:")
print(f"   RMSE: {test_rmse:.4f} km")
print(f"   MAE:  {test_mae:.4f} km")
print(f"   R²:   {test_r2:.4f}")

print("\n5. INTERPRETATION:")
if test_r2 > 0.7:
    print(f"   ✓ Excellent model performance (R² > 0.7)")
elif test_r2 > 0.5:
    print(f"   ✓ Good model performance (R² > 0.5)")
else:
    print(f"   ✗ Model needs improvement (R² < 0.5)")

if abs(train_r2 - test_r2) < 0.1:
    print(f"   ✓ No overfitting detected (train R² ≈ test R²)")
else:
    print(f"   ✗ Possible overfitting (train R² >> test R²)")

print("\n" + "="*70)